# Theorem 20 — triangular stochastic transport

**Formal source:** [`../20_triangular_stochastic_transport.md`](../20_triangular_stochastic_transport.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
rng = np.random.default_rng(20)
count = 30000
common = rng.normal(size=count)
private = rng.normal(size=count)
canonical = 1.5 * common - 0.2
private_after = private.copy()
missing = canonical + 0.5 * private + rng.normal(0, 0.25, count)
conditioned = canonical + 0.5 * private
decoupled = np.zeros(count)
conditioned_mse = np.mean((missing - conditioned) ** 2)
decoupled_mse = np.mean((missing - decoupled) ** 2)
assert np.array_equal(private_after, private)
assert conditioned_mse < 0.08 and conditioned_mse < 0.1 * decoupled_mse
print({"conditioned_missing_mse": float(conditioned_mse), "decoupled_mse": float(decoupled_mse), "private_drift": 0.0})

In [ ]:
print('THEORY_DEMO_PASS::20_triangular_stochastic_transport')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')